In [1]:
print("Hi")

Hi


Introduction to Data Ingestion

In [2]:
import langchain

In [3]:
import os
from typing import List, Dict, Any
import pandas as pd

In [10]:
import langchain
print("LangChain OK")
from langchain_core.documents import Document
print("Core OK")
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("Splitter OK")

LangChain OK
Core OK
Splitter OK


In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)
print("Set up complete")

Set up complete


## Understand the document structure in Langchain

In [12]:

#create a simple document 
doc = Document(
    page_content ="This is the main text content that will be embedded and searched.",
    metadata={
        "source":"example.txt",
        "page":1,
        "author":"Anushka Deokar",
        "date_created ":"2026-07-18",
        "custom_field":"any_value"
    }
)
print("Document Structure")

print(f"Content :{doc.page_content}")
print(f"Content :{doc.metadata}")

Document Structure
Content :This is the main text content that will be embedded and searched.
Content :{'source': 'example.txt', 'page': 1, 'author': 'Anushka Deokar', 'date_created ': '2026-07-18', 'custom_field': 'any_value'}


Metadata is Crucial for :
    Filtering search results
    Tracking document sources
    Providing context in responses
    Debugging and auditing
    ex: if i search who is author of this document 


TextLoader : Read a single file


In [14]:
#from langchain.document_loaders import TextLoader
from langchain_community.document_loaders import TextLoader 

In [18]:
#loading a single text file 
loader = TextLoader("data/text_files/python_intro.txt",encoding="utf-8")

documents =loader.load()
print(type(documents))
print(documents)
print(f"Loaded {len(documents)} documents from the text file.")
print(f"Content preview: {documents[0].page_content[:100]}...")
print(f"Metadata: {documents[0].metadata}")


<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\n    Python is a high-level, interpreted programming language known for its simplicity and readability. \n    Created by Guido van Rossum and first released in 1991, Python has become one of the most popular \n    programming languages in the world.\n\n     Key Features:\n     - Easy to learn and use\n     - Extensive standard library\n     - Cross-platform compatibility\n     - Strong community support\n\n     Python is widely used in various domains, including web development, data analysis, artificial intelligence, scientific computing, and more. Its versatility and ease of use make it a preferred choice for both beginners and experienced developers.\n\n\n    ')]
Loaded 1 documents from the text file.
Content preview: Python Programming Introduction

    Python is a high-level, interpreted programming language known ...
Metadata: {'source': 'data/text_fil

Directory Loader - Mutliple Text Files


In [ ]:
from langchain_community.document_loaders import DirectoryLoader
### load all text files from a directory

dir_loader = DirectoryLoader(
    "data/text_files/",
    glob="**/*.txt", ##pattern to match files folder first then can any files
    loader_cls=TextLoader,  #loader class to use for loading the files
    loader_kwargs ={"encoding":"utf-8"},
    show_progress=True

)

documents =dir_loader.load()

print(f"Loaded {len(documents)} documents from the directory.")
for i , doc in enumerate(documents):
    print(f"Document {i+1}:")
    print(f"Length: {len(doc.page_content[:100])} characters...")
    print(f"Metadata: {doc.metadata}['source']")



    
   

100%|██████████| 2/2 [00:00<00:00, 10.62it/s]

Loaded 2 documents from the directory.
Document 1:
Length: 100 characters...
Metadata: {'source': 'data\\text_files\\machine_learning.txt'}['source']
Document 2:
Length: 100 characters...
Metadata: {'source': 'data\\text_files\\python_intro.txt'}['source']


Directory Loader Characteristics :
Advantages :
    Loads multiple files at once
    Supports global patterns

Disadvantages :
     All files must be of same type
     Limited error handling per file
         

Text splitting strategies

In [22]:
#Different TText splitting strategies
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter,
)
print(documents)




[Document(metadata={'source': 'data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\n    Machine Learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enables systems to learn and improve from experience without being explicitly programmed. \n    It focuses on developing computer programs that can access data and use it to learn for themselves.\n\n    Types of Machine Learning:\n    1. Supervised Learning: Learning from labeled data to make predictions or classify data.\n    2. Unsupervised Learning: Finding patterns and relationships in unlabeled data.\n    3.Reinforcement Learning : Learning through rewards and penalties \n\n    Applications include image recognition, speech processing, and recommendation systems.\n\n    '), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\n    Python is a high-level, interpreted prog

In [23]:
# Method 1 : Character Text Splitter

text = documents[0].page_content
text

'Machine Learning Basics\n\n    Machine Learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enables systems to learn and improve from experience without being explicitly programmed. \n    It focuses on developing computer programs that can access data and use it to learn for themselves.\n\n    Types of Machine Learning:\n    1. Supervised Learning: Learning from labeled data to make predictions or classify data.\n    2. Unsupervised Learning: Finding patterns and relationships in unlabeled data.\n    3.Reinforcement Learning : Learning through rewards and penalties \n\n    Applications include image recognition, speech processing, and recommendation systems.\n\n    '

In [24]:
# 1 . Character Text Splitter

print("Character Text Splitter")
char_splitter = CharacterTextSplitter(
    separator="\n",  # Split on newline characters
    chunk_size=200,   # Maximum chunk size in characters
    chunk_overlap=20, # Overlap between chunks
    length_function=len # how to measure chunk size
)

char_chunks = char_splitter.split_text(text)
print(f"Number of chunks created: {len(char_chunks)}")
print(f"First chunk preview: {char_chunks[0][:100]}...")

Created a chunk of size 229, which is longer than the specified 200


Character Text Splitter
Number of chunks created: 5
First chunk preview: Machine Learning Basics...


In [29]:
print(char_chunks[0])
print("---------")
print(char_chunks[1])
print("---------")
print(char_chunks[2])

Machine Learning Basics
---------
Machine Learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enables systems to learn and improve from experience without being explicitly programmed.
---------
It focuses on developing computer programs that can access data and use it to learn for themselves.
    Types of Machine Learning:


Method 2 : Recursive Character splitiing ( Most Recommended)



In [31]:
print("RECURSIVE CHARACTER TEXT SPLITTER")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],  # List of separators to use for splitting
    chunk_size=200,  # Maximum chunk size in characters
    chunk_overlap=20,  # Overlap between chunks\
    length_function=len
)

recursive_chunks = recursive_splitter.split_text(text)
print(f"Number of chunks created: {len(recursive_chunks)}")
print(f"First chunk preview: {recursive_chunks[0][:100]}...")

RECURSIVE CHARACTER TEXT SPLITTER
Number of chunks created: 7
First chunk preview: Machine Learning Basics...


In [33]:
print(recursive_chunks[0])
print("---------")
print(recursive_chunks[1])

Machine Learning Basics
---------
Machine Learning is a subset of artificial intelligence (AI) that focuses on the development of algorithms and statistical models that enables systems to learn and improve from experience without
